# DFIR-Guardrail: `hard.json` test run (gate v2, phi3:mini)
Step 2 of the frozen protocol in `docs/EVALUATION.md`. Runs `data/hard.json` (132 hand-built evasive payloads, 368 benign false-positive traps) under two conditions, `N_REPEATS` times each:

* **gate_v2**: the default pipeline (rule tier decides alone, prose goes to the LLM, the rest is benign)
* **model_only**: `--no_prefilter`, every artifact goes to the LLM

Same settings as the published runs (T4, `phi3:mini`, `num_ctx=4096`, concurrency 3, daemon restarted per run). Runs with more than 1% inference errors are excluded, because errors are scored as detections. The notebook clones the repo and refuses to run if `src/` differs from the freeze commit, so no bundle upload is needed. Finished runs are skipped on re-run. Run time has not been measured; expect roughly 10 minutes per model_only run.

Run all, then download `hard_results.zip` and unpack it into `results/hard/`.

In [ ]:
# 1. Require a GPU, install Ollama, start the daemon
import subprocess
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then re-run. CPU timeouts invalidate every result.")
!sudo apt-get update -y
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

import os, time, requests
os.environ["OLLAMA_NUM_PARALLEL"] = "3"   # must match CONCURRENCY below
os.environ["CONCURRENCY_LIMIT"] = "3"

def start_ollama():
    global ollama_process
    ollama_process = subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(60):
        try:
            if requests.get("http://localhost:11434").status_code == 200: return
        except Exception:
            time.sleep(1)
    raise RuntimeError("Ollama daemon did not become ready")

def restart_ollama():
    # Fresh daemon per run: requests that timed out in the previous run keep generating server-side and would starve the next one
    ollama_process.kill(); subprocess.run(["pkill", "-f", "ollama"]); time.sleep(3); start_ollama()

start_ollama()
!/usr/local/bin/ollama pull phi3:mini
# Preflight: one real generation, then confirm the model is resident on the GPU
t = time.time()
r = requests.post("http://localhost:11434/api/generate", json={"model": "phi3:mini", "prompt": "Say OK.", "stream": False}, timeout=300)
print(f"preflight generate: {time.time()-t:.1f}s", r.json().get("response", "")[:40])
ps = subprocess.run(["/usr/local/bin/ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
assert "100% GPU" in ps, "phi3:mini is not fully on the GPU; do not run the eval"

In [ ]:
# 2. Clone the frozen code and check that the gate is unchanged since the freeze
REF = "gate-v2-and-eval-harness"   # branch or commit to evaluate
FREEZE = "d378db0"                 # docs/EVALUATION.md: gate v2 frozen here
%cd /content
!rm -rf DFIR-Guardrail && git clone -q https://github.com/YedidyaBarGad/DFIR-Guardrail.git
%cd /content/DFIR-Guardrail
!git checkout -q {REF}
!pip install -q -r requirements.txt
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
changed = subprocess.run(["git", "diff", "--name-only", FREEZE, "HEAD", "--", "src/"], capture_output=True, text=True).stdout.split()
assert not changed, f"src/ changed since the freeze ({changed}); that is a new freeze and needs new held-out data"
print("evaluating commit", COMMIT, "(src/ identical to freeze", FREEZE + ")")

In [ ]:
# 3. Configuration
N_REPEATS = 3
CONCURRENCY = 3
DATASETS = {"hard": "data/hard.json"}
CONDITIONS = {"gate_v2": [], "model_only": ["--no_prefilter"]}

In [ ]:
# 4. Run everything (resumable: a finished run's report is reused)
import json, pathlib
runs = pathlib.Path("runs"); runs.mkdir(exist_ok=True)
rows = []
for ds, ds_file in DATASETS.items():
    for cond, extra in CONDITIONS.items():
        for rep in range(1, N_REPEATS + 1):
            out = runs / f"{ds}_{cond}_{rep}"
            report, log = out / "guardrail_simulation.json", out / "log.txt"
            if not report.exists():
                restart_ollama()
                out.mkdir(exist_ok=True)
                cmd = ["python", "src/pipeline.py", "--mode", "sim", "--concurrency", str(CONCURRENCY),
                       "--dataset_file", ds_file, "--output_dir", str(out), *extra]
                print("RUN", ds, cond, rep, flush=True)
                with open(log, "w") as f:
                    subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env={**os.environ, "PYTHONPATH": "."})
            if not report.exists():
                print("  FAILED, see", log); continue
            m = json.load(open(report))["metadata"]
            rows.append({"dataset": ds, "condition": cond, "run": rep, "scan_s": m["scan_time_seconds"], **m["metrics"],
                         # >1% inference errors means the backend, not the model, produced the labels
                         "valid": m["metrics"]["Errors"] <= 0.01 * m["metrics"]["Total processed"]})
print("done:", len(rows), "runs")
# Record exactly what ran (model digest, Ollama version, gate, embedder status)
print(open(next(iter(runs.glob("*_1/log.txt")))).read().split("--- Run configuration ---")[-1][:600])

In [ ]:
# 5. Per-run table, then mean and min-max range per condition
import math, pandas as pd
df = pd.DataFrame(rows)
df["FNR"] = df["False Negative Rate (FNR)"] * 100
df["FPR"] = df["False Positive Rate (FPR)"] * 100
df.rename(columns={"True Positives":"TP","True Negatives":"TN","False Positives":"FP","False Negatives":"FN"}, inplace=True)
print(df[["dataset","condition","run","TP","TN","FP","FN","Errors","FNR","FPR","scan_s","valid"]].round(2).to_string(index=False))
bad = df[~df.valid]
if len(bad):
    print(f"\nEXCLUDED {len(bad)} invalid run(s) with >1% inference errors (errors are counted as flagged, which fakes detections):")
    print(bad[["dataset","condition","run","Errors"]].to_string(index=False))
df = df[df.valid]

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*2
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return 100*(c-h), 100*(c+h)

summary = []
for (ds, cond), g in df.groupby(["dataset","condition"], sort=False):
    pos = int((g.TP+g.FN).iloc[0])
    lo, hi = wilson(int(g.FN.sum()), int(pos*len(g)))   # pooled over repeats that share one dataset: optimistic
    summary.append({"dataset":ds,"condition":cond,"runs":len(g),
        "FNR mean":g.FNR.mean(),"FNR min":g.FNR.min(),"FNR max":g.FNR.max(),
        "FPR mean":g.FPR.mean(),"FPR min":g.FPR.min(),"FPR max":g.FPR.max(),
        "scan_s mean":g.scan_s.mean(),"errors total":int(g.Errors.sum()),
        "FNR pooled CI lo":lo,"FNR pooled CI hi":hi})
summary = pd.DataFrame(summary).round(2)
print(summary.to_string(index=False))
summary.to_csv("hard_summary.csv", index=False); df.to_csv("hard_runs.csv", index=False)

In [ ]:
# 6. Miss rate (malicious) and false-alarm rate (benign) by hand-built category, valid runs only
cat_of = {d["id"]: d["category"] for d in json.load(open("data/hard.json", encoding="utf-8"))}
rows2 = []
for r in df.itertuples():
    for f in json.load(open(f"runs/{r.dataset}_{r.condition}_{r.run}/guardrail_simulation.json"))["findings"]:
        flagged = not f["verdict"].startswith("Safe")
        # report ids are 1-indexed; a miss on a malicious row, a false alarm on a benign one
        rows2.append({"condition": r.condition, "run": r.run, "category": cat_of[f["id"] - 1],
                      "error": flagged != (f["ground_truth"] == "Malicious")})
by = (pd.DataFrame(rows2).groupby(["condition", "category", "run"]).error.mean().mul(100)
      .groupby(["condition", "category"]).agg(["mean", "min", "max"]).round(1))
print("error % by category (FNR for malicious_*, FPR for benign_*):"); print(by.to_string())
by.to_csv("hard_by_category.csv")

In [ ]:
# 7. Download results and stop Ollama
import shutil
from google.colab import files
open("COMMIT.txt", "w").write(COMMIT + "\n")
for f in ["hard_summary.csv", "hard_runs.csv", "hard_by_category.csv", "COMMIT.txt"]:
    shutil.copy(f, runs / f)
shutil.make_archive("hard_results", "zip", ".", "runs")
files.download("hard_results.zip")
ollama_process.terminate()